# Scaling & Hyperparameter Tuning

![](https://i.ytimg.com/vi/b0L_2jKEbA4/maxresdefault.jpg)

In this notebook you will see a short example of how to scale your data and tune the hyperparameters of your models using grid or randomized search. Along the way you will use cross-validation to evaluate your models more reliably than a single train-test split allows.

Use the raw Titanic CSV included in this repository to compare feature scaling and hyperparameter searches. Imputation and encoding stay **inside each pipeline**, so validation rows do not influence preprocessing.

The older `titanic_preprocessed.csv` remains available for reference, but is not used here because its preprocessing was done before this notebook's split. We keep the same six source features (`Age`, `SibSp`, `Parch`, `Fare`, `Sex`, `Embarked`); the change is when preprocessing is fitted, not which information the models receive.

**Next notebook:** here, cross-validation selects the model within the training set. The heart exercise solution adds a separate validation set to compare candidates after tuning on training folds. Both protocols keep the test set for one final evaluation; a separate validation set is not always required.

## Learning Objectives

By the end of this notebook, you should be able to:

- Scale numerical features with standardization (`StandardScaler`) and min-max scaling (`MinMaxScaler`), and explain when each is useful.
- Combine preprocessing and a model into a single scikit-learn `Pipeline`.
- Evaluate models with K-fold cross-validation instead of relying on one train-test split.
- Tune hyperparameters with `GridSearchCV` and `RandomizedSearchCV`, and compare the trade-offs between them.

In [ ]:
from timeit import default_timer as timer

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RSEED = 10
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RSEED)

In [ ]:
# Load raw data so preprocessing can be learned inside each training fold.
df = pd.read_csv("data/titanic.csv")
df.head()

In [ ]:
# Check for missing data
df.isnull().sum()

## Train-Test Split

We define the target and predictors, then split the dataset into a train and a test set.

In [ ]:
# Keep the target separate and retain the same six source features.
y = df["Survived"]
X = df[["Age", "SibSp", "Parch", "Fare", "Sex", "Embarked"]]

In [ ]:
# Check X
X.head(2)

In [ ]:
# Train-test-split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

---
## Feature Scaling

Often the input features of your model have different units which means that the variables also have different scales. While some model types (e.g. tree-based models like decision tree or random forest) generally do not require scaling of numerical input variables, many machine learning algorithms including for example algorithms using distance measures (e.g. KNN, SVM) can perform better when numerical features are scaled appropriately; improvement is not guaranteed. 

Two common techniques are **min-max scaling** (called normalization in the figure) and **standardization**. Scikit-learn's `Normalizer` instead rescales each row to unit norm; it is not the same as min-max scaling. 

Check the [link](https://machinelearningmastery.com/standardscaler-and-minmaxscaler-transforms-in-python/) for further info. 

![](assets/normalization_vs_standardization.png)

The original dataset in the figure contains observations with different value ranges across the two features. The figure demonstrates two common scaling techniques:

- **Min-max scaling** (called normalization in the figure): Subtract the training minimum, then divide by the training range. For a nonconstant feature, the training minimum becomes 0, the maximum becomes 1, and values in between are mapped proportionally. New values can fall outside [0, 1].
- **Standardization**: Subtract the training mean, then divide by the training standard deviation. Nonconstant training features become centered on 0 with population standard deviation 1. This makes their scales comparable, but does not make their distributions Gaussian.

Both methods preserve each feature's ordering and distribution shape while changing its numerical scale. Different feature-wise rescaling changes distances between observations, which is exactly why KNN predictions can change. Constant features are handled separately; they cannot have variance 1.

In [ ]:
# Route numeric and categorical inputs to different preprocessing steps.
col_scale = ["Age", "SibSp", "Parch", "Fare"]
col_encode = ["Sex", "Embarked"]


def make_preprocessor(scaler="passthrough"):
    # Impute before scaling; passthrough provides the unscaled comparison.
    numeric = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", scaler),
        ]
    )
    # Fill missing categories before encoding; learn both steps on training folds.
    categorical = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )
    return ColumnTransformer(
        [
            ("numeric", numeric, col_scale),
            ("categorical", categorical, col_encode),
        ]
    )

### Standardization

Standardization puts numerical features on a comparable scale. First subtract the training mean to center the values around zero, then divide by the training standard deviation to express their distance from the mean in standard-deviation units. For a nonconstant feature:

$$x_{scaled} = \frac{x-\mu_{train}}{\sigma_{train}}$$

Where:
- $x$ is the individual value to transform.
- $\mu_{train}$ (mu) is the **mean** of the training feature values.
- $\sigma_{train}$ (sigma) is their **standard deviation**, which measures how spread out the values are around the mean.

For $m$ observed training values $x_1, \ldots, x_m$, these statistics are calculated as:

$$\mu_{train} = \frac{1}{m}\sum_{i=1}^{m}x_i$$

$$\sigma_{train} = \sqrt{\frac{1}{m}\sum_{i=1}^{m}(x_i-\mu_{train})^2}$$

Here, $m$ is the number of observed training values, $i$ indexes them, and $\sum$ means to add them together. `StandardScaler` uses this population standard deviation (division by $m$, not $m-1$).

**Example:** If the training mean is 50 and the standard deviation is 10, a value of 70 becomes $(70-50)/10=2$: two standard deviations above the training mean. A value of 50 becomes 0, and a value of 40 becomes -1.

**After transformation:** Nonconstant training features have mean 0 and standard deviation 1. Constant columns become 0 after centering. Standardization preserves distribution shape; it does not make the data Gaussian.

Fit the scaler on training data only; inside cross-validation, fit it separately in each training fold. Reuse those statistics on validation or test data, whose transformed mean and standard deviation need not be 0 and 1.

[`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) is sensitive to outliers.

In [ ]:
# Scale numeric columns while imputing and encoding categorical columns.
scaling_transformation = make_preprocessor(StandardScaler())

### Min-Max Scaling

Min-max scaling (sometimes called normalization) rescales a feature to a chosen range, by default 0 to 1. Subtract the training minimum, then divide by the training range. For a nonconstant feature:

$$x_{scaled} = \frac{x-x_{min,train}}{x_{max,train}-x_{min,train}}$$

Where:
- $x$ is the value to transform.
- $x_{min,train}$ is the smallest observed training value for this feature.
- $x_{max,train}$ is the largest observed training value. Their difference is the training range.

**Example:** With a training minimum of 10 and maximum of 30, a value of 20 becomes $(20-10)/(30-10)=0.5$. A new value of 40 becomes 1.5: do not recompute the minimum and maximum on test data to force it into [0, 1].

[`MinMaxScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html) learns these bounds from training data and reuses them on new observations. Nonconstant training features map to [0, 1]; constant training features map to 0 with the default settings. It does not remove outliers, and extreme training values can compress most other values into a small part of the range.

### Exercise 1: Practice Round

Create the same preprocessing pipeline with min-max scaling instead of standardization. Do not fit it on all data before cross-validation.

<details>
<summary>Show answer</summary>

```python
from sklearn.preprocessing import MinMaxScaler

normalization_transformation = make_preprocessor(MinMaxScaler())
```

`MinMaxScaler` rescales each numerical column using the minimum and maximum learned from training data, mapping its training range to 0-1. Imputation and categorical encoding are unchanged. The pipeline will fit every learned step separately within each training fold.
</details>

In [ ]:
# Scaling with MinMaxScaler

# Try to scale you data with the MinMaxScaler() from sklearn.
# It follows the same syntax as the StandardScaler.
# Don't forget: you have to import the scaler at the top of your notebook.

### Bonus Question

Do standardized or min-max-scaled test values necessarily have the same properties as transformed training values?

For nonconstant training features, that means a mean of 0 and a standard deviation of 1 after standardization, or values between 0 and 1 after min-max scaling.

<details>
<summary>Show answer</summary>

Not necessarily. The scaler learns its parameters from **training data only**, then applies those same fixed parameters to test data:

- **Standardization**: It reuses the training mean and standard deviation. Transformed test features need not have mean 0 or standard deviation 1, or even be close if the distribution has shifted.
- **Min-max scaling**: It reuses the training minimum and maximum. Test values below that minimum or above that maximum fall outside [0, 1], potentially by a large amount.
- **Constant features**: Even in training data, a constant feature cannot have variance 1; with the default centering it becomes 0.

This is expected: reuse the fitted scaler so the model receives the same representation. Fitting a second scaler on the test set would use held-out information and change that representation.
</details>

---
## Predictive Modelling

We will evaluate model performance in a quick and more reliable way using sklearn's [cross_val_score()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html), which implements K-fold cross-validation. When we train a model on a single train-test split we only have one experiment. Can we really trust one experiment?

Think of [K-fold cross-validation](https://en.wikipedia.org/wiki/Cross-validation_(statistics)) as running K experiments and taking the average error. It is still not perfect, but it is more trustworthy than a single split that might turn out unusually good or bad by chance.

This raises the question of which value to use for K. Common values are between 5 and 10, and you also need to account for technical limits: dataset size, compute power, available memory, and time. Cross-validation takes a while on large datasets.

![](assets/cross_validation.png)

For a fair comparison, every candidate below uses the same shuffled stratified five-fold split (`cv`) and accuracy metric. Stratification approximately preserves class proportions; it does not prevent leakage between related passengers such as family members.

The dataset is first split into a **training set** and a **test set**. The test set is held aside and not used during training or model selection.

The training set is then divided into **5 folds**.

Cross-validation works by repeatedly training and validating the model:

* One fold is used as the **validation set**
* The remaining four folds are used as the **training set**
* The model is evaluated on the validation fold to produce one accuracy score in this notebook (higher is better)

This process is repeated **five times**, so that each fold is used as the validation set exactly once.

The mean cross-validation accuracy is calculated as the **average of the five accuracy scores**.

The standard deviation across folds describes score variability, not a confidence interval: the folds have overlapping training data.

### Baseline: Always Predict the Majority Class

Before trying KNN or SGD, establish a simple reference. `DummyClassifier(strategy="most_frequent")` ignores the features and always predicts the most common label in each training fold. Evaluate it using the same folds and accuracy metric as the other models; do not calculate the majority class from the full dataset or the test set.

A model should improve on this reference to justify its complexity. Beating it still does not establish real-world usefulness. [Reference: DummyClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html).

In [ ]:
baseline_scores = cross_val_score(
    DummyClassifier(strategy="most_frequent"),
    X_train,
    y_train,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
)
# Retain fold scores so later examples can be compared without rerunning CV.
cv_scores = {"Majority-class baseline": baseline_scores}
print(f"Baseline mean CV accuracy: {baseline_scores.mean():.3f}")

### KNN Classifier: Unscaled Data

In [ ]:
# Evaluate the complete preprocessing/model pipeline within each CV fold.
knn_classifier = Pipeline(
    [
        ("preprocessing", make_preprocessor()),
        ("knn", KNeighborsClassifier()),
    ]
)
scores = cross_val_score(
    knn_classifier, X_train, y_train, scoring="accuracy", cv=cv, n_jobs=-1
)
cv_scores["KNN - unscaled"] = scores
print(f"Mean accuracy (unscaled): {scores.mean():.3f}")
print(f"Fold standard deviation (unscaled): {scores.std():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=[1, 2, 3, 4, 5], y=scores, ax=ax)
ax.axhline(scores.mean(), color="darkorange", label="Mean accuracy")
ax.set(
    title="KNN: unscaled inputs",
    xlabel="Validation fold",
    ylabel="Accuracy",
    ylim=(0, 1),
)
ax.legend()
fig.tight_layout()

Image description:
Output of the bar plot comparing each of the 5 models trained during cross-validation and corresponding to the results given in the previous cell. Each bar shows one validation-fold accuracy; compare their heights to inspect variability. The horizontal line shows the mean of the five scores. Read the current output rather than expecting fixed values across runs or package versions. The plot is titled 'Scores of the K-Folds Models - unscaled data'.
End of image description

### KNN Classifier: Scaled Data

In [ ]:
# Fit and evaluate using cross-validation; the held-out test is not used.
# Use the same folds and preprocessing choices; change only numeric scaling.
knn_classifier_scaled = Pipeline(
    [
        ("scaling_trans", scaling_transformation),
        ("knn", KNeighborsClassifier()),
    ]
)
scores_scaled = cross_val_score(
    knn_classifier_scaled, X_train, y_train, scoring="accuracy", cv=cv, n_jobs=-1
)
cv_scores["KNN - standardized"] = scores_scaled
print(f"Mean accuracy (scaled): {scores_scaled.mean():.3f}")
print(f"Fold standard deviation (scaled): {scores_scaled.std():.3f}")
print(
    f"Accuracy difference: {100 * (scores_scaled.mean() - scores.mean()):+.1f} percentage points"
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=[1, 2, 3, 4, 5], y=scores_scaled, ax=ax)
ax.axhline(scores_scaled.mean(), color="darkorange", label="Mean accuracy")
ax.set(
    title="KNN: standardized inputs",
    xlabel="Validation fold",
    ylabel="Accuracy",
    ylim=(0, 1),
)
ax.legend()
fig.tight_layout()

Compare the scaled and unscaled mean accuracy and the variability across folds. Scaling may improve performance, but neither the size nor the direction of the change is guaranteed. A change from 0.70 to 0.75 is 5 percentage points, not a 5% relative increase. To show the effect of scaling more clearly, we will look at another classifier that is even more sensitive to it.

### SGD Classifier: Unscaled Data

The [SGD classifier](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html) (stochastic gradient descent classifier) implements a variety of classifiers. The model it fits is controlled by the `loss` parameter: by default it fits a linear support vector machine (SVM). We use it here to emphasise the importance of scaling, and set `loss="log_loss"` so the model becomes a logistic regression trained with stochastic gradient descent.

In [ ]:
# Unscaled comparison: imputation and encoding still happen inside each fold.
sgd_classifier = Pipeline(
    [
        ("preprocessing", make_preprocessor()),
        ("sgd", SGDClassifier(random_state=RSEED, loss="log_loss", max_iter=10000)),
    ]
)
scores = cross_val_score(
    sgd_classifier, X_train, y_train, scoring="accuracy", cv=cv, n_jobs=-1
)
cv_scores["SGD - unscaled"] = scores
print(f"Mean accuracy (unscaled): {scores.mean():.3f}")
print(f"Fold standard deviation (unscaled): {scores.std():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=[1, 2, 3, 4, 5], y=scores, ax=ax)
ax.axhline(scores.mean(), color="darkorange", label="Mean accuracy")
ax.set(
    title="SGD: unscaled inputs",
    xlabel="Validation fold",
    ylabel="Accuracy",
    ylim=(0, 1),
)
ax.legend()
fig.tight_layout()

Inspect how the validation accuracy varies across the different folds. Larger values mean better performance; use the current results rather than assuming a fixed range.

### SGD Classifier: Scaled Data

In [ ]:
# Standardization is fitted on each training fold, not on validation rows.
sgd_classifier_scaled = Pipeline(
    [
        ("scaling_trans", make_preprocessor(StandardScaler())),
        ("sgd", SGDClassifier(random_state=RSEED, loss="log_loss", max_iter=10000)),
    ]
)
scores_scaled = cross_val_score(
    sgd_classifier_scaled, X_train, y_train, scoring="accuracy", cv=cv, n_jobs=-1
)
cv_scores["SGD - standardized"] = scores_scaled
print(f"Mean accuracy (scaled): {scores_scaled.mean():.3f}")
print(f"Fold standard deviation (scaled): {scores_scaled.std():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=[1, 2, 3, 4, 5], y=scores_scaled, ax=ax)
ax.axhline(scores_scaled.mean(), color="darkorange", label="Mean accuracy")
ax.set(
    title="SGD: standardized inputs",
    xlabel="Validation fold",
    ylabel="Accuracy",
    ylim=(0, 1),
)
ax.legend()
fig.tight_layout()

Compare both the mean validation accuracy and its variability across folds before and after scaling. Scaling can help SGD optimization, but improved accuracy and reduced variability are not guaranteed.

### Exercise 2: Practice Round

Try the KNN classifier with the min-max-scaled data.

<details>
<summary>Show answer</summary>

Reuse the normalization transformer from Exercise 1 inside a `Pipeline`, just as we did for the standardized KNN model, then evaluate it with cross-validation:

```python
knn_normalized = Pipeline(
    [("scaling_trans", normalization_transformation), ("knn", KNeighborsClassifier())]
)
scores_norm = cross_val_score(
    knn_normalized, X_train, y_train, scoring="accuracy", cv=cv, n_jobs=-1
)
print("Score (normalized):", round(scores_norm.mean(), 2))
cv_scores["KNN - min-max"] = scores_norm
```

Both scalers change distances, but they need not give similar results. Compare them on the same validation folds; neither is universally better.
</details>

In [ ]:
# Train model using the MinMaxScaler data you scaled in ex1.

# It follows the same syntax as the StandardScaler.

### Scaling Comparison at a Glance

Collect the previous fold scores in one table. The gain is the difference from the baseline in **percentage points**, not a relative percentage. If you completed Exercise 2 using the answer above, the min-max result also appears. No extra models are trained here.

Which model improves on the baseline most? How does scaling affect KNN versus SGD? Read the fold standard deviation as variability, not as a confidence interval or proof of a statistically significant difference. This table compares untuned models; the KNN searches below are a separate demonstration.

In [ ]:
cv_comparison = pd.DataFrame(
    {
        name: {
            "mean_cv_accuracy": fold_scores.mean(),
            "fold_standard_deviation": fold_scores.std(),
            "gain_vs_baseline_pp": 100 * (fold_scores.mean() - baseline_scores.mean()),
        }
        for name, fold_scores in cv_scores.items()
    }
).T
cv_comparison.round(3)

---
## Hyperparameter Tuning

Most models have many hyperparameters, and the best values differ from one dataset to another. The same is true for the regularization hyperparameters we saw earlier, which are usually chosen by trial and error. So how do we find the parameter values that work best for our data?

#### GridSearchCV

[Grid search](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) is a tuning technique that searches for the best hyperparameter values. It performs an exhaustive search over a predefined parameter space using cross-validation (hence the **CV** suffix), evaluating every possible combination in the search space to find and return the best one.

This becomes time-consuming when there are many hyperparameters and the search space is large. For example, with k=5 and 2 parameters taking 2 and 3 values respectively (6 combinations), GridSearchCV runs 30 modelling steps just to find the best values for those two parameters.

![](assets/grid_search_cv.png)

The process evaluates multiple combinations of hyperparameters to identify the model configuration that performs best.

In this example, two hyperparameters are considered:

* parameter **a** with possible values **1, 2, and 3**
* parameter **b** with possible values **1 and 2**

This produces **6 possible hyperparameter combinations**.

For each combination, **5-fold cross-validation** is performed:

* the training data is divided into 5 folds
* the model is trained and validated 5 times
* the performance scores are averaged to obtain a cross-validation score for that parameter combination

This process is repeated for all 6 combinations.

In total, **30 cross-validation fits are performed (6 × 5)**. With the default `refit=True`, the best configuration is then fitted once more on all the training data supplied to the search.

The parameter combination with the best average validation performance is selected as the final model configuration.


In [ ]:
# what hyperparameters does KNN have?
knn_classifier_scaled.get_params()

In [ ]:
# Defining parameter grid (as dictionary)
param_grid = {
    "knn__n_neighbors": [2, 3, 4, 5, 10],  # number of neighbors
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2, 3],
}

# Instantiate gridsearch and define the metric to optimize
gs = GridSearchCV(
    knn_classifier_scaled, param_grid, scoring="accuracy", cv=cv, verbose=0, n_jobs=-1
)

# Fit gridsearch object to data.. also lets see how long it takes
start = timer()
gs.fit(X_train, y_train)
end = timer()
gs_time = end - start

In [ ]:
gs_time

In [ ]:
# Best score
print("Best score:", round(gs.best_score_, 3))

# Best parameters
print("Best parameters:", gs.best_params_)

In [ ]:
# Keep the reporting logic together; call it only for the final selected model.
def print_pretty_summary(name, best_model, y_test, y_pred_test):
    print(name)
    print("=======================")
    model = best_model.steps[-1][1]
    print(f"n_neighbors: {model.n_neighbors}")
    print(f"weights: {model.weights}")
    print(f"p: {model.p}")
    print(f"algorithm: {model.algorithm}")

    accuracy = accuracy_score(y_test, y_pred_test)
    print(f"Final test accuracy: {accuracy:.3f}")
    return accuracy

In [ ]:
knn_best = gs.best_estimator_
knn_best

In [ ]:
# Keep the test set untouched until both searches have finished.
# Compare cross-validation scores, not test scores, to select the model.
print(f"Grid search best CV accuracy: {gs.best_score_:.3f}")

#### RandomizedSearchCV

As an alternative to grid search we can use sklearn's [RandomizedSearchCV()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html). Random search does not try every possible combination in the search space; instead it randomly picks and evaluates a set number of parameter combinations.

In [ ]:
# Define parameter grid for randomized search
param_grid = {
    "knn__n_neighbors": [2, 3, 4, 5, 10],  # number of neighbors
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2, 3],
}
# Instantiate random search and define the metric to optimize
rs = RandomizedSearchCV(
    knn_classifier_scaled,
    param_grid,
    scoring="accuracy",
    cv=cv,
    n_iter=10,
    verbose=0,
    n_jobs=-1,
    random_state=RSEED,
)

# Fit randomized search object to data
start = timer()
rs.fit(X_train, y_train)
end = timer()
rgs_time = end - start

In [ ]:
# Best score
print("Best score:", round(rs.best_score_, 3))

# Best parameters
print("Best parameters:", rs.best_params_)

In [ ]:
searches = {"Grid search": gs, "Randomized search": rs}
comparison = pd.DataFrame(
    {
        "best_cv_accuracy": [gs.best_score_, rs.best_score_],
        "candidates": [len(gs.cv_results_["params"]), len(rs.cv_results_["params"])],
        "elapsed_seconds": [gs_time, rgs_time],
    },
    index=list(searches),
)
comparison

In [ ]:
selected_name = max(searches, key=lambda name: searches[name].best_score_)
selected_model = searches[selected_name].best_estimator_
y_pred_test = selected_model.predict(X_test)
print(f"Selected by CV accuracy: {selected_name}")
final_test_accuracy = print_pretty_summary(
    selected_name, selected_model, y_test, y_pred_test
)

## Interpretation

The final evaluation uses the model already refitted on all training data by the chosen search. In an exact CV tie, the code selects grid search because it appears first in the predefined dictionary.

Randomized search evaluates 10 of the 30 listed candidates here; grid search evaluates all 30. Runtime depends on hardware and caching, so one timing is not a benchmark. Do not use the test score to go back and tune again. Accuracy is suitable for this introductory comparison but does not describe every kind of classification error.

## Export for Notebook 3

After selection and final evaluation, save the fitted pipeline and a small metadata file. This reuses `selected_model` exactly as evaluated; there is no new training or tuning. Notebook 3 loads these files in a fresh kernel to predict on invented requests.

The files stay in the Git-ignored `artifacts/` directory. Save the complete pipeline, including preprocessing. [Reference: scikit-learn model persistence](https://scikit-learn.org/stable/model_persistence.html).

In [ ]:
import json
from importlib.metadata import version
from pathlib import Path

import joblib

artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)
fitted_preprocessor = selected_model.steps[0][1]
fitted_encoder = fitted_preprocessor.named_transformers_["categorical"].named_steps[
    "encoder"
]
model_info = {
    "selected_search": selected_name,
    "input_features": list(X_train.columns),
    "numeric_features": col_scale,
    "categorical_features": col_encode,
    "known_categories": {
        column: categories.tolist()
        for column, categories in zip(
            col_encode, fitted_encoder.categories_, strict=True
        )
    },
    "numeric_training_ranges": X_train[col_scale].agg(["min", "max"]).to_dict(),
    "package_versions": {
        package: version(package)
        for package in ["scikit-learn", "numpy", "pandas", "joblib"]
    },
    "evaluation": {
        "cv_accuracy": float(searches[selected_name].best_score_),
        "test_accuracy": float(final_test_accuracy),
    },
}
# Persist the fitted model; the metadata describes its inputs and existing evaluation.
joblib.dump(selected_model, artifact_dir / "titanic_pipeline.joblib")
(artifact_dir / "titanic_model_info.json").write_text(
    json.dumps(model_info, indent=2) + "\n"
)
print("Saved the fitted pipeline and metadata for notebook 3 in artifacts/.")